In [1]:
#|default_exp utils

In [2]:
#| export
import sys
sys.path.append('..')
import os
import numpy as np
import torch
import random
from scipy.stats import kendalltau
from tpuoptimize.fastai_fit import Metric



In [3]:
#| export
def good_luck():
    return True

In [4]:
# | export 
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


def listMLE(y_pred, y_true, eps=1e-10, padded_value_indicator=-1):
    """
    ListMLE loss introduced in "Listwise Approach to Learning to Rank - Theory and Algorithm".
    :param y_pred: predictions from the model, shape [batch_size, slate_length]
    :param y_true: ground truth labels, shape [batch_size, slate_length]
    :param eps: epsilon value, used for numerical stability
    :param padded_value_indicator: an indicator of the y_true index containing a padded item, e.g. -1
    :return: loss value, a torch.Tensor
    """
    # shuffle for randomised tie resolution
    random_indices = torch.randperm(y_pred.shape[-1])
    y_pred_shuffled = y_pred[:, random_indices]
    y_true_shuffled = y_true[:, random_indices]

    y_true_sorted, indices = y_true_shuffled.sort(descending=True, dim=-1)

    mask = y_true_sorted == padded_value_indicator

    preds_sorted_by_true = torch.gather(y_pred_shuffled, dim=1, index=indices)
    preds_sorted_by_true[mask] = float("-inf")

    max_pred_values, _ = preds_sorted_by_true.max(dim=1, keepdim=True)

    preds_sorted_by_true_minus_max = preds_sorted_by_true - max_pred_values

    cumsums = torch.cumsum(preds_sorted_by_true_minus_max.exp().flip(dims=[1]), dim=1).flip(dims=[1])

    observation_loss = torch.log(cumsums + eps) - preds_sorted_by_true_minus_max

    observation_loss[mask] = 0.0

    return torch.mean(torch.sum(observation_loss, dim=1))

def listMLE_loss(x,y):
    return listMLE(x['pred'],y['config_runtime'])


def rank_loss(p,y):
    ids = y['config_runtime'].view(-1).sort()[1]
    pred = p['pred'].view(-1)[ids]
    B = pred.shape[0]
    dist = pred[None,:] - pred[:,None]
    
    #pos = dist[torch.split(torch.triu_indices(B,B,1), 1, dim=0)].flatten()
    neg = dist[torch.split(torch.tril_indices(B,B,1), 1, dim=0)].flatten()
    #loss = torch.cat([F.softplus(-pos,1), F.softplus(neg,1)])
    loss = F.softplus(neg,1)
    
    loss = loss.mean() + 1e-3*torch.pow(dist,2).mean() #make sure that predictions do not diverge 
    
    return loss

def loss_tot(p,y):
    return listMLE_loss(p,y) + 10*rank_loss(p,y)

class KTC(Metric):
    def __init__(self, idx=-1):
        super().__init__()
        self.reset()
        self.idx = idx
        
    def reset(self): 
        self.buf = []
        
    def accumulate(self,learn):
        if self.idx != -1 and learn.y['data_idx'].view(-1)[0].item() != self.idx: return
        self.buf.append(kendalltau(learn.pred['pred'].view(-1).cpu(),
                                   learn.y['config_runtime'].view(-1).cpu(), variant = 'c')[0])
    
    @property
    def value(self):
        if len(self.buf) == 0: self.buf = [0]
        return torch.FloatTensor(self.buf).mean()
    
class KTC_mean(Metric):
    def __init__(self, idx=-1):
        super().__init__()
        self.metrics = [KTC(idx=i) for i in range(4)]
        
    def reset(self): 
        for i in range(len(self.metrics)): self.metrics[i].reset()
            
    def accumulate(self,learn):
        for i in range(len(self.metrics)): self.metrics[i].accumulate(learn)
       
    @property
    def value(self):     
        return torch.FloatTensor([m.value for m in self.metrics]).mean()

In [5]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()